# OpenUnlearning — LLaMA 3.2 3B Benchmarking
**Survey: Unified Benchmarking of LLM Unlearning Methods**

Run cells in order. Checkpoints auto-save to Google Drive.

> **Note:** Run Cell 3b (Torch Fix) first, then restart the runtime before continuing from Cell 2.

## Cell 1 — Check GPU

In [ ]:
!nvidia-smi
import torch
print(f'CUDA: {torch.cuda.is_available()}')
print(f'GPU: {torch.cuda.get_device_name(0)}')
print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

## Cell 2 — Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!mkdir -p /content/drive/MyDrive/open-unlearning-saves/unlearn
!mkdir -p /content/drive/MyDrive/open-unlearning-saves/eval
print('Drive mounted and save dirs ready.')

## Cell 3 — Clone Repo & Install Dependencies

In [ ]:
import os

if not os.path.exists('/content/open-unlearning'):
    !git clone https://github.com/locuslab/open-unlearning.git /content/open-unlearning
else:
    print('Repo already exists, skipping clone.')

%cd /content/open-unlearning
!pip install ".[lm-eval]" -q
!pip install --no-build-isolation flash-attn==2.6.3 -q
print('Installation complete.')

## Cell 3b — Fix Torch/Torchvision Mismatch
**Run this cell, then restart the runtime (Runtime → Restart session) before continuing.**

In [ ]:
# Fix torchvision/torch version mismatch
# Colab ships a newer torch that is incompatible with OpenUnlearning
!pip install torch==2.4.1 torchvision==0.19.1 torchaudio==2.4.1 \
  --index-url https://download.pytorch.org/whl/cu121 -q
print('Torch reinstall complete.')
print('NOW RESTART THE RUNTIME: Runtime → Restart session')
print('Then run cells 2, 4, 5, 6, 7, 8, 9 in order.')

## Cell 4 — Fix BFloat16 Bug (required for 3B on A100)

In [ ]:
!sed -i 's/avg_losses = avg_losses.cpu().numpy().tolist()/avg_losses = avg_losses.cpu().float().numpy().tolist()/g' \
    /content/open-unlearning/src/evals/metrics/utils.py

!sed -i 's/normalized_probs = normalized_probs.cpu().numpy().tolist()/normalized_probs = normalized_probs.cpu().float().numpy().tolist()/g' \
    /content/open-unlearning/src/evals/metrics/utils.py

# Verify both fixes applied
!grep 'cpu().float()' /content/open-unlearning/src/evals/metrics/utils.py
print('BFloat16 fix applied.')

## Cell 5 — Download Eval Baselines

In [ ]:
%cd /content/open-unlearning
!python setup_data.py --eval
print('Eval baselines downloaded.')

## Cell 6 — HuggingFace Login

In [ ]:
from huggingface_hub import login
login()  # Paste your HF token (needs Read access to meta-llama models)

## Cell 7 — Restore Checkpoints from Drive
*(Skip on first run — no checkpoints exist yet)*

In [ ]:
import os

drive_saves = '/content/drive/MyDrive/open-unlearning-saves'
local_saves = '/content/open-unlearning/saves'
os.makedirs(local_saves, exist_ok=True)

if os.path.exists(f'{drive_saves}/unlearn'):
    !cp -rn {drive_saves}/unlearn {local_saves}/
    print('Restored unlearn checkpoints from Drive.')
else:
    print('No unlearn checkpoints on Drive yet.')

if os.path.exists(f'{drive_saves}/eval'):
    !cp -rn {drive_saves}/eval {local_saves}/
    print('Restored eval results from Drive.')
else:
    print('No eval results on Drive yet.')

print('\nCheckpoints available:')
!ls {local_saves}/unlearn/ 2>/dev/null || echo 'None'
print('\nEvals available:')
!ls {local_saves}/eval/ 2>/dev/null || echo 'None'

## Cell 8 — Write run_colab.sh Script

In [ ]:
script = '''
#!/bin/bash
set -e

METHODS=("GradAscent" "GradDiff" "NPO" "RMU" "SimNPO" "UNDIAL" "WGA" "SatImp")
MODEL="Llama-3.2-3B-Instruct"
TASK_SUFFIX="tofu_3B"
FORGET_SPLIT="forget10"
RETAIN_SPLIT="retain90"
DRIVE_SAVES="/content/drive/MyDrive/open-unlearning-saves"

cd /content/open-unlearning

for METHOD in "${METHODS[@]}"; do
  TASK_NAME="${METHOD}_${TASK_SUFFIX}"
  CKPT_DIR="saves/unlearn/${TASK_NAME}"
  EVAL_DIR="saves/eval/${TASK_NAME}_eval"

  echo "==============================="
  echo " $METHOD"
  echo "==============================="

  if [ -d "$CKPT_DIR" ] && [ -f "${CKPT_DIR}/config.json" ]; then
    echo "  [SKIP] checkpoint exists"
  else
    echo "  [TRAIN]"
    python src/train.py --config-name=unlearn.yaml \\
      experiment=unlearn/tofu/default \\
      forget_split=${FORGET_SPLIT} retain_split=${RETAIN_SPLIT} \\
      trainer=${METHOD} task_name=${TASK_NAME} \\
      model=${MODEL} || { echo "  [FAIL] train failed"; continue; }
    cp -r ${CKPT_DIR} ${DRIVE_SAVES}/unlearn/ 2>/dev/null || true
  fi

  if [ -f "${EVAL_DIR}/TOFU_SUMMARY.json" ]; then
    echo "  [SKIP] eval exists"
  else
    echo "  [EVAL]"
    python src/eval.py --config-name=eval.yaml \\
      experiment=eval/tofu/default \\
      forget_split=${FORGET_SPLIT} \\
      task_name=${TASK_NAME}_eval \\
      model.model_args.pretrained_model_name_or_path=${CKPT_DIR} \\
      || { echo "  [FAIL] eval failed"; continue; }
    cp -r ${EVAL_DIR} ${DRIVE_SAVES}/eval/ 2>/dev/null || true
  fi

  echo "  [DONE] $METHOD"
done

echo ""
echo "==============================="
echo " RESULTS SUMMARY"
echo "==============================="
for METHOD in "${METHODS[@]}"; do
  SUMMARY="saves/eval/${METHOD}_${TASK_SUFFIX}_eval/TOFU_SUMMARY.json"
  echo "=== $METHOD ==="
  cat "$SUMMARY" 2>/dev/null || echo "MISSING"
done
'''

with open('/content/open-unlearning/run_colab.sh', 'w') as f:
    f.write(script)

!chmod +x /content/open-unlearning/run_colab.sh
print('run_colab.sh written and ready.')

## Cell 9 — Run All Methods (train + eval)
**~4-5 hours total. Already-completed methods are skipped automatically.**

Run the keep-alive script first, then the training script.

In [ ]:
%%javascript
function ClickConnect(){
    console.log('Keeping alive...');
    document.querySelector('colab-toolbar-button#connect').click()
}
setInterval(ClickConnect, 60000)

In [ ]:
import os
os.environ['TQDM_DISABLE'] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

!nohup bash /content/open-unlearning/run_colab.sh > \
  /content/drive/MyDrive/open-unlearning-saves/run_colab.log 2>&1 &
print('Running in background. Monitor with Cell 9b.')

## Cell 9b — Monitor Progress
*Run this anytime to check training status.*

In [ ]:
!tail -30 /content/drive/MyDrive/open-unlearning-saves/run_colab.log

In [ ]:
# Check if process is still running
!ps aux | grep run_colab | grep -v grep || echo 'Process not running'

## Cell 10 — View Final Results

In [ ]:
import json, os

methods = ['GradAscent', 'GradDiff', 'NPO', 'RMU', 'SimNPO', 'UNDIAL', 'WGA', 'SatImp']
results = {}

for method in methods:
    path = f'/content/open-unlearning/saves/eval/{method}_tofu_3B_eval/TOFU_SUMMARY.json'
    if os.path.exists(path):
        with open(path) as f:
            results[method] = json.load(f)
    else:
        results[method] = None

print(f'{"Method":<15} {"Extr.Str↓":>10} {"ForgetProb↓":>12} {"ROUGE↓":>8} {"Utility↑":>10} {"Privleak":>10}')
print('-' * 70)
for method, r in results.items():
    if r:
        print(f'{method:<15} {r["extraction_strength"]:>10.3f} {r["forget_Q_A_Prob"]:>12.3f} {r["forget_Q_A_ROUGE"]:>8.3f} {r["model_utility"]:>10.3f} {r["privleak"]:>10.1f}')
    else:
        print(f'{method:<15} {"MISSING":>55}')

## Cell 11 — Manual Backup to Drive
*Run anytime to save progress.*

In [ ]:
!cp -r /content/open-unlearning/saves/unlearn /content/drive/MyDrive/open-unlearning-saves/ 2>/dev/null || true
!cp -r /content/open-unlearning/saves/eval /content/drive/MyDrive/open-unlearning-saves/ 2>/dev/null || true
print('Backup complete.')
!ls /content/drive/MyDrive/open-unlearning-saves/